In [1]:
import pandas as pd
import os 
import allel
import random
import subprocess
import pickle
from os import path as op
import numpy as np

import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

/home/tbellagio/.local/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


The inputs for this projects are:
- The terminal generation for grenenet
- The bioclim data across the 3 years the experiment ran for 

Asuumptions: with this project we are assuming that after the 3 generations in the grenenet experiment, the populations became locally adapted to the environemnt 
so we are trying to find differences in between the different adapted subpopulations and their relationship to climate. 




For baypass in particular we would need:
- omega with neutral loci 
- The number of flowers collected: pool sizes
- The allele frequencies 
- The environmental variables 

In [2]:
flowers = pd.read_csv('/home/tbellagio/safedata/meixilin/grenenet/metadata/data/samples_data.csv')

In [3]:
flowers = pd.read_csv('/home/tbellagio/safedata/meixilin/grenenet/metadata/data/merged_samples_data.csv')

In [4]:
flowers['site'] = flowers['sample_name'].str.split('_').str[0]
flowers['gen'] = flowers['sample_name'].str.split('_').str[1]
flowers['plot'] = flowers['sample_name'].str.split('_').str[2]

In [5]:
final_gen = flowers.groupby(['site', 'plot'])['generation'].max().reset_index()

In [6]:
final_gen = flowers.merge(final_gen, on = ['site', 'plot','generation'])

In [7]:
#final_gen.to_csv('final_gen.csv')

In [8]:
#Choose SNPs for GEA structure correction in baypass

In [9]:
#os.listdir('/home/tbellagio/safedata/meixilin/grenenet/metadata/data')

In [10]:
clim_sites_during_exp = pd.read_csv('clim_sites_during_exp.csv')

In [129]:
#pd.read_csv('/home/tbellagio/safedata/ath_evo/grenephase1-data/merged_tables/merged_hapFIRE_allele_frequency_LDpruned.txt', sep = '\t')

In [13]:
#all_samples_af = os.listdir('/carnegie/nobackup/scratch/xwu/grenet/hapFIRE_frequencies/samples/snp_frequency/')

In [22]:
## the size of this file is 3235481 * 745  
path_all_af = '/carnegie/nobackup/scratch/xwu/grenet/merged_frequency/merged_hapFIRE_allele_frequency.csv'

In [23]:
## path to ld prunned allele freq
path_ldp_af = '/carnegie/nobackup/scratch/xwu/grenet/merged_frequency/merged_hapFIRE_allele_frequency_LDpruned.txt'

In [36]:
samples = pd.read_csv(path_ldp_af, sep = '\t',nrows=1).columns

## for now lets just work with final generation
samples = [value for value in samples if value in final_gen['sample_name'].tolist()]

In [37]:
# Read the file with only the specified columns
df_ldp = dd.read_csv(path_ldp_af, sep = '\t', usecols=samples)

In [38]:
df_shape = df_ldp.shape

In [39]:
# The number of rows needs to be computed; the number of columns is immediate
num_rows = df_shape[0].compute()  # This computes the actual number of rows
num_columns = df_shape[1] 

In [40]:
num_rows

13985

In [41]:
num_columns

60

In [35]:
## get the vcf file frothe chromosomes and positions 

## ld pruned vcf file
ld_prunned_vcf_file = '/carnegie/nobackup/scratch/xwu/grenet/hapFIRE_updatedVCF/greneNet_final_v1.1_LDpruned.recode.vcf'
ld_prunned_vcf = allel.read_vcf(ld_prunned_vcf_file)